Importing All useful Libraries

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

In [2]:
train_meta = pd.read_csv("../input/siim-isic-melanoma-classification/train.csv")
test_meta = pd.read_csv("../input/siim-isic-melanoma-classification/test.csv")

useful_cols = ['sex',"age_approx","anatom_site_general_challenge"]
TARGET = "target"
ID = "image_name"
train_meta = train_meta[useful_cols+[TARGET,ID]]

In [3]:
train_meta.isna().sum()

sex                               65
age_approx                        68
anatom_site_general_challenge    527
target                             0
image_name                         0
dtype: int64

We see that we have some Nan values in repective columns , so as test data also contains anatom_site_general_challenge wit some nan values ,hence we fill a new class into these nan positions and drop records where age or sex are provided as nan.

In [4]:
train_meta['anatom_site_general_challenge'] = train_meta['anatom_site_general_challenge'].fillna("unknown_site")
test_meta['anatom_site_general_challenge'] = test_meta['anatom_site_general_challenge'].fillna("unknown_site")

In [5]:
train_meta.dropna(inplace=True)

We use LabelEncoder to encode the string columns present in our dataset. and store respective encoders for each column into a dictionary so that it can be used in test data also.

In [6]:
from sklearn.preprocessing import LabelEncoder
from collections import defaultdict

encoders = defaultdict(LabelEncoder)
for column in train_meta.select_dtypes("object").columns:
    if column in [ID,TARGET]:
        continue
    encoder = LabelEncoder()
    train_meta[column] = encoder.fit_transform(train_meta[[column]])
    encoders[column] = encoder

/opt/conda/lib/python3.7/site-packages/sklearn/utils/validation.py:73: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  return f(**kwargs)


In [7]:
train_meta.shape

(33058, 5)

Now comes the main model part.
I have implemented 5 kfold ,to examine my model performance.

In [8]:
X = train_meta.drop([ID,TARGET],axis=1)
Y = train_meta[[TARGET]]

from sklearn.model_selection import KFold
folds = KFold(n_splits=5,shuffle=True)

params ={
    "od_type":"Iter",
    'od_wait':100,
    "eval_metric":"AUC",
    'loss_function':'Logloss',
    "iterations":1000,
    "verbose":100
}

scores = []

max_score = -np.inf
for (train_idx,test_idx),i in zip(folds.split(X,Y),range(0,5)):
    print("Working On fold ",i)
    model = CatBoostClassifier(**params)
    model.fit(X.iloc[train_idx],Y.iloc[train_idx],
              eval_set=(X.iloc[test_idx],Y.iloc[test_idx]),
              cat_features = ["sex","anatom_site_general_challenge"])
    
    score = model.score(model.predict(X.iloc[test_idx]),Y.iloc[test_idx])
    print("Achieved AUC Score :" ,score)
    scores.append(score)
    print(scores)
    if score > max_score:
        best_idx = (train_idx,test_idx)
        max_score = score
        
    print("-"*100)
    

print("Final Results from 5 KFOLD")
print("Min Score",min(scores))
print("Mean Score",sum(scores)/len(scores))
print("Max Score",max(scores))

Working On fold  0
Learning rate set to 0.071159
0:	test: 0.6340869	best: 0.6340869 (0)	total: 81.8ms	remaining: 1m 21s
100:	test: 0.6587332	best: 0.6592478 (38)	total: 1.75s	remaining: 15.6s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.659247763
bestIteration = 38

Shrink model to first 39 iterations.
Achieved AUC Score : 0.9844222625529341
[0.9844222625529341]
----------------------------------------------------------------------------------------------------
Working On fold  1
Learning rate set to 0.071159
0:	test: 0.6830524	best: 0.6830524 (0)	total: 18.7ms	remaining: 18.7s
100:	test: 0.7083900	best: 0.7092545 (91)	total: 1.53s	remaining: 13.6s
200:	test: 0.7089723	best: 0.7100999 (148)	total: 3.12s	remaining: 12.4s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.7100998796
bestIteration = 148

Shrink model to first 149 iterations.
Achieved AUC Score : 0.9804900181488203
[0.9844222625529341, 0.9804900181488203]
--------------------------

I have used the best indices provided from above experiment to prepare my model for final submission

In [9]:
model = CatBoostClassifier(**params)
model.fit(X.iloc[best_idx[0]],Y.iloc[best_idx[0]],
              eval_set=(X.iloc[best_idx[1]],Y.iloc[best_idx[1]]),
              cat_features = ["sex","anatom_site_general_challenge"])

score = model.score(model.predict(X.iloc[test_idx]),Y.iloc[test_idx])
print("Achieved AUC Score :" ,score)
    

Learning rate set to 0.071159
0:	test: 0.6340869	best: 0.6340869 (0)	total: 19.1ms	remaining: 19.1s
100:	test: 0.6587332	best: 0.6592478 (38)	total: 1.72s	remaining: 15.3s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.659247763
bestIteration = 38

Shrink model to first 39 iterations.
Achieved AUC Score : 0.9816971713810316


## Submission Time

converting classes back to there respective encoded values using LabelEncoders we trained before.

In [10]:
for key in encoders.keys():
    test_meta[key] = encoders[key].transform(test_meta[key])
test_meta.head()

,image_name,patient_id,sex,age_approx,anatom_site_general_challenge
0,ISIC_0052060,IP_3579794,1,70.0,5
1,ISIC_0052349,IP_7782715,1,40.0,1
2,ISIC_0058510,IP_7960270,0,55.0,4
3,ISIC_0073313,IP_6375035,0,50.0,4
4,ISIC_0073502,IP_0589375,0,45.0,1


Final Touch to submit  csv file.

In [11]:
testing = test_meta.drop([ID,'patient_id'],axis=1)
predictions = model.predict_proba(testing)
predictions = [i[1] for i in predictions]
test_meta[TARGET] = predictions
test_meta[[ID,TARGET]].to_csv("catboost_submission.csv",index=None)

With this code , i was able to achieve 0.7 score on public leaderboard.
I know this is not much to be used , but atleast it can be useful for some one.

Will be updating this notebook after hyperparameter tuning .

Kindly comment if you like or dislike something in this notebook, and if you lked please upvote it too.
and also do share some suggestions which can be used to improve it.

Thank You.